In [1]:
import os
import requests
import pandas as pd
from io import BytesIO
from dotenv import load_dotenv

load_dotenv()


class APIClient:
    """Client pour récupérer les données Vélib et météo."""

    VELIB_URL = (
        "https://opendata.paris.fr/api/explore/v2.1/"
        "catalog/datasets/velib-disponibilite-en-temps-reel/"
        "exports/csv?use_labels=true"
    )

    METEO_URL = (
        "https://public-api.meteofrance.fr/public/DPObs/v2/"
        "station/infrahoraire-6m?id_station="
    )

    STATIONS_METEO_URL = (
        "https://public-api.meteofrance.fr/public/DPObs/v2/liste-stations"
    )

    CITY_DEPARTEMENT_CODE = "75"

    def __init__(self, timeout: int = 30):
        self.timeout = timeout
        self.session = requests.Session()
        self.token = os.getenv("TOKEN")
        self.dataset_url = None  # Initialisation propre

    def get_city_stations_id(self):
        response = self.session.get(
            self.STATIONS_METEO_URL,
            headers={
                "accept": "*/*",
                "apikey": self.token,
            },
            timeout=self.timeout,
        )
        response.raise_for_status()

        stations_df = pd.read_csv(
            BytesIO(response.content),
            encoding="utf-8-sig",
            sep=None,
            engine="python",
            dtype={"Id_station": str},
        )
        
        id_stations_df = stations_df[stations_df["Id_station"].str.startswith("75", na=False)]["Id_station"]

        return id_stations_df

    def get_velib_dataset(self):
        response = self.session.get(
            self.dataset_url,
            timeout=self.timeout,
        )
        response.raise_for_status()

        return pd.read_csv(
            BytesIO(response.content),
            encoding="utf-8-sig",
            sep=None,
            engine="python",
        )

    def get_meteo_dataset(self):
        if not self.token:
            raise ValueError(
                "TOKEN absent du fichier .env ou non chargé."
            )

        list_stations_id = self.get_city_stations_id()
        dataframes = []

        for station in list_stations_id:
            url = f"{self.dataset_url}{station}&format=json"

            response = self.session.get(
                url,
                headers={
                    "accept": "*/*",
                    "apikey": self.token,
                },
                timeout=self.timeout,
            )

            response.raise_for_status()
            data = pd.json_normalize(response.json())
            dataframes.append(data)

        if dataframes:
            return pd.concat(dataframes, ignore_index=True)

        return pd.DataFrame()

    def get_dataset(self, dataset_name: str) -> pd.DataFrame:
        """Récupère les données et les convertit en DataFrame."""
        if dataset_name == "velib":
            self.dataset_url = self.VELIB_URL
            dataset = self.get_velib_dataset()

        elif dataset_name == "meteo":
            self.dataset_url = self.METEO_URL
            dataset = self.get_meteo_dataset()

        else:
            raise ValueError(
                "Dataset inconnu. Choisissez 'velib' ou 'meteo'."
            )

        return dataset

    def close(self) -> None:
        self.session.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

    @staticmethod
    def fetch_dataset(dataset: str) -> pd.DataFrame:
        with APIClient() as client:
            return client.get_dataset(dataset)

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

True

In [3]:
with APIClient() as client:
    df_velib = client.get_dataset("velib")
print(f"Stations chargées : {len(df_velib)}")

Stations chargées : 1526


In [4]:
print(df_velib.columns.tolist())

['Identifiant station', 'Nom station', 'Station en fonctionnement', 'Capacité de la station', 'Nombre bornettes libres', 'Nombre total vélos disponibles', 'Vélos mécaniques disponibles', 'Vélos électriques disponibles', 'Borne de paiement disponible', 'Retour vélib possible', 'Actualisation de la donnée', 'Coordonnées géographiques', 'Nom communes équipées', 'Code INSEE communes équipées', 'station_opening_hours']


In [5]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
token = os.getenv("TOKEN")

if token:
    print(f"Token lu (longueur : {len(token)} caractères) : {token[:10]}...")
else:
    print("❌ Aucun token trouvé.")

Token lu (longueur : 2234 caractères) : eyJ4NXQiOi...


In [6]:
headers = {
    "accept": "application/json",
    "apikey": token
}

In [7]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

# 1. Charger le fichier .env depuis la racine du projet
load_dotenv(dotenv_path="../.env")
token = os.getenv("TOKEN")

if not token:
    print("❌ Erreur : Le TOKEN est introuvable dans le fichier .env.")
else:
    print(f"✅ Token chargé avec succès ! (Longueur : {len(token)} caractères)")

    # 2. En-têtes spécifiques pour le protocole API Key
    headers = {
        "accept": "application/json",
        "apikey": token  # <-- C'est ici que se situait l'ajustement crucial !
    }

    # 3. URL de l'API DPClim (exemple pour la liste des stations de Paris)
    url = "https://public-api.meteofrance.fr/public/DPClim/v1/liste-stations/quotidienne?id-departement=75"

    # 4. Requête GET
    response = requests.get(url, headers=headers)

    # 5. Affichage du résultat
    if response.status_code == 200:
        print("🌤️ Données récupérées avec succès !")
        data = response.json()
        df = pd.DataFrame(data)
        display(df.head())
    else:
        print(f"⚠️ Erreur {response.status_code} : {response.text}")

✅ Token chargé avec succès ! (Longueur : 2234 caractères)
🌤️ Données récupérées avec succès !


,id,nom,posteOuvert,typePoste,lon,lat,alt,postePublic
0,75101001,INNOCENTS,False,4,2.348333,48.860667,37,True
1,75101002,MONNAIE,False,4,2.336667,48.868333,33,True
2,75104001,TOUR ST-JACQUES,False,4,2.348333,48.858333,38,True
3,75105001,PLANTES,False,4,2.363333,48.843333,35,True
4,75105004,PARIS-5E BASSIN DU PANTHEON,False,4,2.348333,48.845000,68,True


In [10]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
token = os.getenv("TOKEN", "").strip()

if not token:
    print("❌ Aucun TOKEN trouvé dans ../.env")
else:
    headers = {
        "accept": "*/*",
        "apikey": token,
    }

    base_url = "https://public-api.meteofrance.fr/public/DPPaquetObs/v2"

    # 1. Test simple : liste des stations d'observation
    url_stations = f"{base_url}/liste-stations"
    r_stations = requests.get(url_stations, headers=headers)
    print("1) /liste-stations -> Statut :", r_stations.status_code)
    if r_stations.status_code == 200:
        print("Aperçu stations :", r_stations.text[:300])
    else:
        print("Réponse stations :", r_stations.text[:300])

    # 2. Test paquet horaire (ajoute les paramètres requis si l'API renvoie 400)
    url_horaire = f"{base_url}/paquet/horaire"
    r_horaire = requests.get(url_horaire, headers=headers)
    print("\n2) /paquet/horaire -> Statut :", r_horaire.status_code)
    if r_horaire.status_code == 200:
        with open("paquet_horaire.csv", "wb") as f:
            f.write(r_horaire.content)
        print("✅ Fichier sauvegardé sous paquet_horaire.csv")
    else:
        print("Réponse horaire :", r_horaire.text[:300])

1) /liste-stations -> Statut : 200
Aperçu stations : Id_station;Id_omm;Nom_usuel;Latitude;Longitude;Altitude;Date_ouverture;Pack
01014002;;ARBENT;46.278167;5.669000;534;2003-10-01;RADOME
01027003;;BALAN_AERO;45.833000;5.106667;196;2014-05-26;ETENDU
01034004;;BELLEY;45.769333;5.688000;330;2001-09-13;RADOME
01064001;;VERIZIEU;45.777167;5.487167;281;2015

2) /paquet/horaire -> Statut : 400
Réponse horaire : <am:fault xmlns:am="http://wso2.org/apimanager"><am:code>400</am:code><am:type>Rapport d'erreur</am:type><am:message>Contrôle de paramètres en erreur</am:message><am:description>La valeur "" est incorrecte. Vérifiez le paramètre format.</am:description></am:fault>


In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
token = os.getenv("TOKEN", "").strip()

headers = {
    "accept": "*/*",
    "apikey": token,
}

url_horaire = "https://public-api.meteofrance.fr/public/DPPaquetObs/v2/paquet/horaire"
params = {
    "id-departement": "75",
    "format": "csv",  # ou "json" si tu préfères parser du JSON directement
}

r_horaire = requests.get(url_horaire, headers=headers, params=params)
print("Statut :", r_horaire.status_code)

if r_horaire.status_code == 200:
    ext = "csv" if params["format"] == "csv" else "json"
    nom_fichier = f"paquet_horaire_75.{ext}"
    with open(nom_fichier, "wb") as f:
        f.write(r_horaire.content)
    print(f"✅ Fichier sauvegardé sous {nom_fichier}")
    print(r_horaire.text[:500])
else:
    print("Réponse :", r_horaire.text)

In [19]:
import pandas as pd

df_meteo = pd.read_csv("paquet_horaire_75.csv", sep=";")

# Conversion des températures principales de Kelvin en °C
for col in ["t", "tx", "tn"]:
    if col in df_meteo.columns:
        df_meteo[f"{col}_c"] = df_meteo[col] - 273.15

df_meteo["validity_dt"] = pd.to_datetime(df_meteo["validity_time"])

colonnes_utiles = [
    c
    for c in ["geo_id_insee", "validity_dt", "t_c", "tx_c", "tn_c", "rr1", "ff", "u"]
    if c in df_meteo.columns
]
display(df_meteo[colonnes_utiles].head(10))

,geo_id_insee,validity_dt,t_c,tx_c,tn_c,rr1,ff,u
0,75106001,2026-11-21 18:00:00+00:00,9.8,9.8,9.5,0.0,NaN,NaN
1,75106001,2026-11-21 17:00:00+00:00,9.8,10.5,9.8,0.0,NaN,NaN
2,75106001,2026-11-21 16:00:00+00:00,10.5,11.2,10.5,0.0,NaN,NaN
3,75106001,2026-11-21 15:00:00+00:00,11.2,11.5,11.2,0.0,NaN,NaN
4,75106001,2026-11-21 14:00:00+00:00,11.5,11.6,11.3,0.0,NaN,NaN
5,75106001,2026-11-21 13:00:00+00:00,11.4,11.4,11.2,0.0,NaN,NaN
6,75106001,2026-11-21 12:00:00+00:00,11.2,11.2,11.1,0.0,NaN,NaN
7,75106001,2026-11-21 11:00:00+00:00,11.1,11.1,10.8,0.0,NaN,NaN
8,75106001,2026-11-21 10:00:00+00:00,10.8,10.8,10.2,0.0,NaN,NaN
9,75106001,2026-11-21 09:00:00+00:00,10.1,10.1,9.8,0.0,NaN,NaN


In [20]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
token = os.getenv("TOKEN", "").strip()

headers = {
    "accept": "*/*",
    "apikey": token,
}

url_stations = "https://public-api.meteofrance.fr/public/DPPaquetObs/v2/liste-stations"
response = requests.get(url_stations, headers=headers)

print("Statut :", response.status_code)

if response.status_code == 200:
    nom_fichier = "liste_stations_obs.csv"
    with open(nom_fichier, "wb") as f:
        f.write(response.content)
    print(f"✅ Fichier sauvegardé sous {nom_fichier}")

    df_stations = pd.read_csv(nom_fichier, sep=";", dtype={"Id_station": str})
    # Filtrer les stations commençant par '75' (Paris)
    df_75 = df_stations[df_stations["Id_station"].str.startswith("75", na=False)]
    display(df_75)
else:
    print("Erreur :", response.text)

Statut : 200
✅ Fichier sauvegardé sous liste_stations_obs.csv


,Id_station,Id_omm,Nom_usuel,Latitude,Longitude,Altitude,Date_ouverture,Pack
1552,75106001,NaN,LUXEMBOURG,48.844833,2.338500,50,1955-03-01,ETENDU
1553,75107005,NaN,TOUR EIFFEL,48.858333,2.294500,330,1996-07-01,ETENDU
1554,75110001,NaN,LARIBOISIERE,48.882833,2.352000,55,1912-12-01,ETENDU
1555,75114001,7156.0,PARIS-MONTSOURIS,48.821667,2.337833,75,1872-06-17,RADOME
1556,75114007,NaN,PARIS-MONTSOURIS-DOUBLE,48.821667,2.337833,75,2016-06-02,ETENDU
1557,75116008,NaN,LONGCHAMP,48.854833,2.233667,27,2007-03-01,RADOME
